In [ ]:
import pandas as pd
from pathlib import Path

# ----------------------------
# CONFIG
# ----------------------------

RAW_FILE = "C:\\Users\\honor\\Downloads\\GSS_stata\\gss7224_r2.dta"          # change to your GSS file
CACHE_FILE = "gss_cached.parquet" # fast reload file

CORE_VARS = [
    "year",
    "happy",
    "life",
    "health",
    "goodlife"
]

DEP_VARS = ["happy", "life", "health", "goodlife"]


# ----------------------------
# FUNCTIONS
# ----------------------------

def load_gss(path=RAW_FILE, usecols=None):
    return pd.read_csv(path, usecols=usecols, encoding="latin1")


def subset_years(df, min_year=1996):
    return df.loc[df["year"] >= min_year].copy()


def standardize_within_year(df, cols):
    df = df.copy()
    for c in cols:
        z = f"{c}_z"
        df[z] = df.groupby("year")[c].transform(
            lambda x: (x - x.mean()) / x.std(ddof=0)
        )
    return df


def build_depression_index(df):
    df = standardize_within_year(df, DEP_VARS)

    zcols = [f"{c}_z" for c in DEP_VARS]

    # reverse so higher = more depression
    df[zcols] = -df[zcols]

    df["depression_index"] = df[zcols].mean(axis=1)

    return df


def prepare_gss(force_rebuild=False):
    """
    Loads cached processed dataset if available.
    Otherwise builds it from raw GSS and caches it.
    """

    if Path(CACHE_FILE).exists() and not force_rebuild:
        print("Loading cached dataset...")
        return pd.read_parquet(CACHE_FILE)

    print("Processing raw GSS dataset...")

    df = load_gss(usecols=CORE_VARS)
    df = subset_years(df, 1996)
    df = build_depression_index(df)

    df.to_parquet(CACHE_FILE, index=False)

    print("Dataset cached for future runs.")

    return df


# ----------------------------
# RUN PIPELINE
# ----------------------------

df_96 = prepare_gss()

print(df_96.shape)
df_96.head()

Processing raw GSS dataset...


UnicodeDecodeError: 'utf-8' codec can't decode byte 0xf8 in position 70: invalid start byte